# Image 1

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import gc
import time

# ============================================================
# BAR CHART WITH NUMBERS
# Employment Trend for Top Occupation Groups, 2010-2025
#
# INPUT DATA:
# /Users/YourUserName123/Downloads/OldJob/all_2010-2025_job_CLEANED_numeric_only_3.csv
#
# OUTPUT IMAGE:
# /Users/YourUserName123/Downloads/05_Employment_trend_for_top_occupation_groups_BAR_with_numbers.png
# ============================================================

INPUT_FILE = (
    Path.home()
    / "Downloads"
    / "OldJob"
    / "all_2010-2025_job_CLEANED_numeric_only_3.csv"
)

OUTPUT_FILE = (
    Path.home()
    / "Downloads"
    / "05_Employment_trend_for_top_occupation_groups_BAR_with_numbers.png"
)

CHUNKSIZE = 150_000

use_cols = [
    "year",
    "occ_code",
    "occ_title",
    "tot_emp"
]

print("=" * 100)
print("INPUT FILE")
print(INPUT_FILE)
print("=" * 100)

if not INPUT_FILE.exists():
    raise FileNotFoundError(f"File not found: {INPUT_FILE}")

print("OUTPUT FILE")
print(OUTPUT_FILE)
print("=" * 100)

# ============================================================
# READ FILE MEMORY OPTIMIZED
# ============================================================

parts = []
rows_seen = 0
start_time = time.time()

for i, chunk in enumerate(
    pd.read_csv(
        INPUT_FILE,
        usecols=use_cols,
        chunksize=CHUNKSIZE,
        low_memory=False
    ),
    start=1
):
    rows_seen += len(chunk)

    # Clean year
    chunk["year"] = pd.to_numeric(chunk["year"], errors="coerce")

    # Clean employment number
    chunk["tot_emp"] = (
        chunk["tot_emp"]
        .astype(str)
        .str.replace(",", "", regex=False)
        .replace(["*", "**", "#", "nan", "NaN", "None", ""], pd.NA)
    )
    chunk["tot_emp"] = pd.to_numeric(chunk["tot_emp"], errors="coerce")

    # Clean text columns
    chunk["occ_code"] = chunk["occ_code"].astype(str).str.strip()
    chunk["occ_title"] = chunk["occ_title"].astype(str).str.strip()

    # Drop bad rows
    chunk = chunk.dropna(subset=["year", "occ_code", "occ_title", "tot_emp"])
    chunk["year"] = chunk["year"].astype(int)

    # Keep only 2010-2025
    chunk = chunk[(chunk["year"] >= 2010) & (chunk["year"] <= 2025)]

    # Keep only major occupation groups, example:
    # 11-0000, 13-0000, 15-0000, 29-0000
    chunk = chunk[chunk["occ_code"].str.match(r"^\d{2}-0000$", na=False)]

    parts.append(chunk)

    if i == 1 or i % 5 == 0:
        kept_so_far = sum(len(x) for x in parts)
        elapsed = (time.time() - start_time) / 60
        print(
            f"Chunk {i} done | rows seen: {rows_seen:,} | "
            f"major group rows kept: {kept_so_far:,} | "
            f"minutes: {elapsed:.1f}"
        )

    del chunk
    gc.collect()

df = pd.concat(parts, ignore_index=True)

del parts
gc.collect()

print("=" * 100)
print("READ DONE")
print("Rows kept before dedup:", len(df))
print("=" * 100)

# ============================================================
# DEDUPLICATE
# Do NOT sum duplicates.
# Keep ONE best row per year + occupation group by highest employment.
# ============================================================

df = df.sort_values(
    ["year", "occ_code", "tot_emp"],
    ascending=[True, True, False]
)

df = df.drop_duplicates(
    subset=["year", "occ_code"],
    keep="first"
)

print("Rows after dedup:", len(df))
print("Years found:", sorted(df["year"].unique()))
print("=" * 100)

# ============================================================
# CHOOSE TOP 10 OCCUPATION GROUPS BY 2025 EMPLOYMENT
# ============================================================

latest_year = 2025

top_groups = (
    df[df["year"] == latest_year]
    .sort_values("tot_emp", ascending=False)
    .head(10)["occ_title"]
    .tolist()
)

print("TOP 10 GROUPS USED:")
for n, name in enumerate(top_groups, start=1):
    print(n, name)

plot_df = df[df["occ_title"].isin(top_groups)].copy()

# Shorten names so legend is easier to read
plot_df["occ_title_short"] = (
    plot_df["occ_title"]
    .str.replace("Occupations", "Occ.", regex=False)
    .str.replace("and", "&", regex=False)
)

# Pivot for grouped bar chart
pivot = plot_df.pivot_table(
    index="year",
    columns="occ_title_short",
    values="tot_emp",
    aggfunc="first"
).sort_index()

print("=" * 100)
print("DATA USED FOR CHART")
print(pivot)
print("=" * 100)

# ============================================================
# MAKE BAR CHART WITH NUMBERS
# ============================================================

plt.figure(figsize=(26, 12))

ax = pivot.plot(
    kind="bar",
    figsize=(26, 12),
    width=0.85
)

plt.title(
    "Employment Trend for Top Occupation Groups, 2010-2025",
    fontsize=20,
    fontweight="bold"
)

plt.xlabel("Year", fontsize=13)
plt.ylabel("Employment", fontsize=13)
plt.xticks(rotation=45)
plt.grid(axis="y", alpha=0.3)

# Add numbers on top of bars
for container in ax.containers:
    labels = []
    for value in container.datavalues:
        if pd.isna(value) or value == 0:
            labels.append("")
        else:
            labels.append(f"{value/1_000_000:.1f}M")

    ax.bar_label(
        container,
        labels=labels,
        fontsize=7,
        rotation=90,
        padding=2
    )

plt.legend(
    title="Occupation group",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()

# Save directly to Downloads
plt.savefig(OUTPUT_FILE, dpi=300, bbox_inches="tight")

plt.show()

print("=" * 100)
print("DONE")
print("Image saved to:")
print(OUTPUT_FILE)
print("=" * 100)

# ver 2 side way

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import gc
import time

# ============================================================
# SIDEWAY / HORIZONTAL BAR CHART WITH NUMBERS
# Employment Trend for Top Occupation Groups, 2010-2025
#
# INPUT:
# /Users/YourUserName123/Downloads/OldJob/all_2010-2025_job_CLEANED_numeric_only_3.csv
#
# OUTPUT:
# /Users/YourUserName123/Downloads/05_Employment_trend_for_top_occupation_groups_SIDEWAY_BAR_with_numbers.png
# ============================================================

INPUT_FILE = (
    Path.home()
    / "Downloads"
    / "OldJob"
    / "all_2010-2025_job_CLEANED_numeric_only_3.csv"
)

OUTPUT_FILE = (
    Path.home()
    / "Downloads"
    / "05_Employment_trend_for_top_occupation_groups_SIDEWAY_BAR_with_numbers.png"
)

CHUNKSIZE = 150_000

use_cols = [
    "year",
    "occ_code",
    "occ_title",
    "tot_emp"
]

print("=" * 100)
print("INPUT FILE")
print(INPUT_FILE)
print("=" * 100)

if not INPUT_FILE.exists():
    raise FileNotFoundError(f"File not found: {INPUT_FILE}")

print("OUTPUT FILE")
print(OUTPUT_FILE)
print("=" * 100)

# ============================================================
# READ FILE MEMORY OPTIMIZED
# ============================================================

parts = []
rows_seen = 0
start_time = time.time()

for i, chunk in enumerate(
    pd.read_csv(
        INPUT_FILE,
        usecols=use_cols,
        chunksize=CHUNKSIZE,
        low_memory=False
    ),
    start=1
):
    rows_seen += len(chunk)

    chunk["year"] = pd.to_numeric(chunk["year"], errors="coerce")

    chunk["tot_emp"] = (
        chunk["tot_emp"]
        .astype(str)
        .str.replace(",", "", regex=False)
        .replace(["*", "**", "#", "nan", "NaN", "None", ""], pd.NA)
    )
    chunk["tot_emp"] = pd.to_numeric(chunk["tot_emp"], errors="coerce")

    chunk["occ_code"] = chunk["occ_code"].astype(str).str.strip()
    chunk["occ_title"] = chunk["occ_title"].astype(str).str.strip()

    chunk = chunk.dropna(subset=["year", "occ_code", "occ_title", "tot_emp"])
    chunk["year"] = chunk["year"].astype(int)

    chunk = chunk[(chunk["year"] >= 2010) & (chunk["year"] <= 2025)]

    # Keep only major occupation groups
    chunk = chunk[chunk["occ_code"].str.match(r"^\d{2}-0000$", na=False)]

    parts.append(chunk)

    if i == 1 or i % 5 == 0:
        kept_so_far = sum(len(x) for x in parts)
        elapsed = (time.time() - start_time) / 60
        print(
            f"Chunk {i} done | rows seen: {rows_seen:,} | "
            f"major group rows kept: {kept_so_far:,} | "
            f"minutes: {elapsed:.1f}"
        )

    del chunk
    gc.collect()

df = pd.concat(parts, ignore_index=True)

del parts
gc.collect()

print("=" * 100)
print("READ DONE")
print("Rows kept before dedup:", len(df))
print("=" * 100)

# ============================================================
# DEDUPLICATE
# Do NOT sum duplicates.
# Keep ONE best row per year + occupation group by highest employment.
# ============================================================

df = df.sort_values(
    ["year", "occ_code", "tot_emp"],
    ascending=[True, True, False]
)

df = df.drop_duplicates(
    subset=["year", "occ_code"],
    keep="first"
)

print("Rows after dedup:", len(df))
print("Years found:", sorted(df["year"].unique()))
print("=" * 100)

# ============================================================
# CHOOSE TOP 10 OCCUPATION GROUPS BY 2025 EMPLOYMENT
# ============================================================

latest_year = 2025

top_groups = (
    df[df["year"] == latest_year]
    .sort_values("tot_emp", ascending=False)
    .head(10)["occ_title"]
    .tolist()
)

print("TOP 10 GROUPS USED:")
for n, name in enumerate(top_groups, start=1):
    print(n, name)

plot_df = df[df["occ_title"].isin(top_groups)].copy()

plot_df["occ_title_short"] = (
    plot_df["occ_title"]
    .str.replace("Occupations", "Occ.", regex=False)
    .str.replace("and", "&", regex=False)
)

pivot = plot_df.pivot_table(
    index="year",
    columns="occ_title_short",
    values="tot_emp",
    aggfunc="first"
).sort_index()

print("=" * 100)
print("DATA USED FOR CHART")
print(pivot)
print("=" * 100)

# ============================================================
# MAKE SIDEWAY / HORIZONTAL BAR CHART WITH NUMBERS
# ============================================================

ax = pivot.plot(
    kind="barh",
    figsize=(28, 14),
    width=0.85
)

plt.title(
    "Employment Trend for Top Occupation Groups, 2010-2025",
    fontsize=20,
    fontweight="bold"
)

plt.xlabel("Employment", fontsize=13)
plt.ylabel("Year", fontsize=13)
plt.grid(axis="x", alpha=0.3)

# Add numbers at end of bars
for container in ax.containers:
    labels = []
    for value in container.datavalues:
        if pd.isna(value) or value == 0:
            labels.append("")
        else:
            labels.append(f"{value/1_000_000:.1f}M")

    ax.bar_label(
        container,
        labels=labels,
        fontsize=7,
        padding=3
    )

plt.legend(
    title="Occupation group",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()

plt.savefig(OUTPUT_FILE, dpi=300, bbox_inches="tight")

plt.show()

print("=" * 100)
print("DONE")
print("Image saved to:")
print(OUTPUT_FILE)
print("=" * 100)

# ver 3 

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import gc
import time

# ============================================================
# OVERALL TOTAL EMPLOYMENT BY YEAR
# 2010-2025
#
# This makes ONE total number for each year.
# It uses 00-0000 / All Occupations if available.
# If not available, it sums major occupation groups.
#
# INPUT:
# /Users/YourUserName123/Downloads/OldJob/all_2010-2025_job_CLEANED_numeric_only_3.csv
#
# OUTPUT:
# /Users/YourUserName123/Downloads/01_Overall_total_employment_2010_2025_SIDEWAY_BAR_with_numbers.png
# ============================================================

INPUT_FILE = (
    Path.home()
    / "Downloads"
    / "OldJob"
    / "all_2010-2025_job_CLEANED_numeric_only_3.csv"
)

OUTPUT_FILE = (
    Path.home()
    / "Downloads"
    / "01_Overall_total_employment_2010_2025_SIDEWAY_BAR_with_numbers.png"
)

CHUNKSIZE = 150_000

use_cols = [
    "year",
    "occ_code",
    "occ_title",
    "tot_emp"
]

print("=" * 100)
print("INPUT FILE")
print(INPUT_FILE)
print("=" * 100)

if not INPUT_FILE.exists():
    raise FileNotFoundError(f"File not found: {INPUT_FILE}")

print("OUTPUT FILE")
print(OUTPUT_FILE)
print("=" * 100)

# ============================================================
# READ FILE MEMORY OPTIMIZED
# ============================================================

parts = []
rows_seen = 0
start_time = time.time()

for i, chunk in enumerate(
    pd.read_csv(
        INPUT_FILE,
        usecols=use_cols,
        chunksize=CHUNKSIZE,
        low_memory=False
    ),
    start=1
):
    rows_seen += len(chunk)

    # Clean year
    chunk["year"] = pd.to_numeric(chunk["year"], errors="coerce")

    # Clean employment number
    chunk["tot_emp"] = (
        chunk["tot_emp"]
        .astype(str)
        .str.replace(",", "", regex=False)
        .replace(["*", "**", "#", "nan", "NaN", "None", ""], pd.NA)
    )
    chunk["tot_emp"] = pd.to_numeric(chunk["tot_emp"], errors="coerce")

    # Clean text
    chunk["occ_code"] = chunk["occ_code"].astype(str).str.strip()
    chunk["occ_title"] = chunk["occ_title"].astype(str).str.strip()

    # Drop bad rows
    chunk = chunk.dropna(subset=["year", "occ_code", "occ_title", "tot_emp"])
    chunk["year"] = chunk["year"].astype(int)

    # Keep only 2010-2025
    chunk = chunk[(chunk["year"] >= 2010) & (chunk["year"] <= 2025)]

    parts.append(chunk)

    if i == 1 or i % 5 == 0:
        elapsed = (time.time() - start_time) / 60
        print(
            f"Chunk {i} done | rows seen: {rows_seen:,} | "
            f"kept rows: {sum(len(x) for x in parts):,} | "
            f"minutes: {elapsed:.1f}"
        )

    del chunk
    gc.collect()

df = pd.concat(parts, ignore_index=True)

del parts
gc.collect()

print("=" * 100)
print("READ DONE")
print("Rows kept:", len(df))
print("=" * 100)

# ============================================================
# METHOD 1:
# Use 00-0000 / All Occupations
# Keep highest employment row per year.
# This usually gives national overall employment.
# ============================================================

all_occ = df[
    (df["occ_code"] == "00-0000") |
    (df["occ_title"].str.lower().eq("all occupations"))
].copy()

if len(all_occ) > 0:
    print("Using method: 00-0000 / All Occupations")

    total_by_year = (
        all_occ
        .sort_values(["year", "tot_emp"], ascending=[True, False])
        .drop_duplicates(subset=["year"], keep="first")
        [["year", "tot_emp"]]
        .sort_values("year")
        .reset_index(drop=True)
    )

else:
    # ============================================================
    # METHOD 2 FALLBACK:
    # If 00-0000 is missing, sum major occupation groups.
    # Keep one best row per year + occupation group first.
    # ============================================================

    print("00-0000 not found.")
    print("Using fallback method: sum major occupation groups.")

    major = df[df["occ_code"].str.match(r"^\d{2}-0000$", na=False)].copy()
    major = major[major["occ_code"] != "00-0000"]

    major = major.sort_values(
        ["year", "occ_code", "tot_emp"],
        ascending=[True, True, False]
    )

    major = major.drop_duplicates(
        subset=["year", "occ_code"],
        keep="first"
    )

    total_by_year = (
        major
        .groupby("year", as_index=False)["tot_emp"]
        .sum()
        .sort_values("year")
        .reset_index(drop=True)
    )

# ============================================================
# PRINT DATA USED
# ============================================================

print("=" * 100)
print("OVERALL TOTAL EMPLOYMENT BY YEAR")
print("=" * 100)

print(
    total_by_year.assign(
        total_employment=total_by_year["tot_emp"].map(lambda x: f"{x:,.0f}")
    )[["year", "total_employment"]].to_string(index=False)
)

print("=" * 100)

# ============================================================
# MAKE SIDEWAY BAR CHART WITH NUMBERS
# ============================================================

plot_df = total_by_year.copy()
plot_df["year"] = plot_df["year"].astype(str)

fig, ax = plt.subplots(figsize=(14, 10))

bars = ax.barh(
    plot_df["year"],
    plot_df["tot_emp"]
)

ax.set_title(
    "Overall Total Employment by Year, 2010-2025",
    fontsize=18,
    fontweight="bold"
)

ax.set_xlabel("Total Employment")
ax.set_ylabel("Year")
ax.grid(axis="x", alpha=0.3)

# Add number labels
for bar in bars:
    value = bar.get_width()
    ax.text(
        value,
        bar.get_y() + bar.get_height() / 2,
        f" {value/1_000_000:.1f}M",
        va="center",
        fontsize=10
    )

plt.tight_layout()

plt.savefig(OUTPUT_FILE, dpi=300, bbox_inches="tight")

plt.show()

print("=" * 100)
print("DONE")
print("Image saved to:")
print(OUTPUT_FILE)
print("=" * 100)

# ver 4

In [ ]:
# ============================================================
# MAKE SIDEWAY BAR CHART
# Different color for each sector
# IMAGE ONLY
# ============================================================

from matplotlib.ticker import FuncFormatter

plot_df = summary.copy()

plot_df["sector_short"] = (
    plot_df["occ_title"]
    .str.replace("Occupations", "Occ.", regex=False)
    .str.replace("and", "&", regex=False)
)

# Sort ascending so biggest shows at top in horizontal chart
plot_df = plot_df.sort_values(
    "total_employment_2010_2025",
    ascending=True
).reset_index(drop=True)

plt.figure(figsize=(18, 12))

# Different color for every bar
cmap = plt.get_cmap("tab20")
colors = [cmap(i % cmap.N) for i in range(len(plot_df))]

bars = plt.barh(
    plot_df["sector_short"],
    plot_df["total_employment_2010_2025"],
    color=colors,
    edgecolor="black",
    linewidth=0.4
)

plt.title(
    "Overall Total Employment by Occupation Sector, 2010-2025",
    fontsize=18,
    fontweight="bold"
)

plt.xlabel("Total Employment, 2010-2025")
plt.ylabel("Occupation Sector")

plt.grid(axis="x", alpha=0.3)

# Format x-axis as millions
plt.gca().xaxis.set_major_formatter(
    FuncFormatter(lambda x, pos: f"{x/1_000_000:.0f}M")
)

# Add extra space on right for number labels
max_value = plot_df["total_employment_2010_2025"].max()
plt.xlim(0, max_value * 1.15)

# Add numbers to end of bars
for bar in bars:
    value = bar.get_width()
    plt.text(
        value,
        bar.get_y() + bar.get_height() / 2,
        f" {value/1_000_000:.1f}M",
        va="center",
        fontsize=9
    )

plt.tight_layout()
plt.savefig(OUTPUT_IMAGE, dpi=300, bbox_inches="tight")
plt.show()

print("=" * 100)
print("DONE")
print("Image saved to:")
print(OUTPUT_IMAGE)
print("=" * 100)